# **計算例**
## 計算数理I（理学部）・計算数理（教養学部）2026年度Sセメスター
東京大学大学院数理科学研究科　齊藤宣一　[https://norikazu-saito.github.io/p/](https://norikazu-saito.github.io/p/) [最新更新日：2026.05.31]

次は必ず実行してください

In [ ]:
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')
# インポート
import numpy as np
import matplotlib.pyplot as plt

# 1. 非線形方程式

図2.1（例2.4）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

theta = np.linspace(0, 2*np.pi, 1000)
xc = np.cos(theta)
yc = np.sin(theta)
x = np.linspace(-2, 2, 2000)
y = -np.cbrt(np.sin(np.pi * x / 2))

plt.figure(figsize=(6, 6))
plt.plot(xc, yc, label=r"$x^2+y^2=1$", linewidth=2)
plt.plot(x, y,label=r"$\sin(\pi x/2)+y^3=0$",linewidth=2)
plt.gca().set_aspect('equal')
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True, linestyle="--", alpha=0.5)
#plt.legend()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/example2d.pdf',dpi=300, bbox_inches='tight')
plt.show()

図2.2（例2.4）

In [ ]:
def newton_mult(f, dfdx, x, eps, maxiter):
  fval = f(x)
  xvect = np.copy(x)
  fvect = np.copy(fval)
  magnitude = np.linalg.norm(fval, ord=np.inf)
  while magnitude > eps and iter < maxiter:
    dfdxval = dfdx(x)
    if np.abs(np.linalg.det(dfdxval)) < 1.0e-15:
      print('Jacobi matrix is almost singular for x = ', x)
      sys.exit(1)
    w = np.linalg.solve(dfdxval, fval)
    x -= w
    fval = f(x)
    magnitude = np.linalg.norm(fval, ord=np.inf)
    iter += 1
    xvect=np.vstack((xvect,x))
    fvect=np.vstack((fvect,fval))
  return xvect, fvect, iter

In [ ]:
def newton_draw(xvect):
  yvect = xvect[1:,:] - xvect[:-1,:]
  plt.quiver(xvect[:-1,0], xvect[:-1,1], yvect[:,0], yvect[:,1], angles='xy', scale_units='xy', scale=1,color='b')
  plt.plot(xvect[:,0], xvect[:,1],'co',ms=5)
  plt.grid('on')
  plt.gca().set_aspect('equal', adjustable='box')
  plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/mnewton.pdf',dpi=300, bbox_inches='tight')
  plt.show()
  return 0

In [ ]:
def f(x):
  return np.array([x[0]**2.0 + x[1]**2.0 - 1.0, np.sin(np.pi*x[0]/2.0)+x[1]**3.0])

def dfdx(x):
  return np.array([[2.0*x[0], 2.0*x[1]],
             [np.pi/2.0*np.cos(np.pi*x[0]/2.0),3.0*x[1]**2.0]])

kmax=20
x=np.array([-1.0,3.0])
#x=np.array([5.0,-2.0])
#x=np.array([7.0,3.3])

eps=1e-12
xvect, fvect, iter= newton_mult(f, dfdx, x, eps, kmax)

print(f'max number of iterations: {kmax:d}')
print('iteration',iter)
print(xvect)

newton_draw(xvect)

図2.3（例2.9）

In [ ]:
def durand_kerner(f, n, beta, r, maxiter, eps):
  # Aberth の方法による初期値の設定
  gamma=1.7
  t = np.linspace(0,n-1,n)
  z = beta+r*np.exp(1.0j*(2*np.pi*t/n+gamma))
  # Newton iteration
  iter = 0
  zvect = np.array([z])
  f_val = f(z)
  err = np.linalg.norm(f_val, ord=np.inf)
  while err > eps and iter < maxiter:
    df_val = np.prod(z.reshape(-1,1) - z.reshape(1, -1) + np.eye(n,n) , axis=1)
    if np.linalg.norm(df_val, ord=np.inf)<1.0e-15:
      print('denominator is almost zero')
      sys.exit(1)
    z -= f_val / df_val
    f_val = f(z)
    err = np.linalg.norm(f_val, ord=np.inf)
    iter += 1
    zvect = np.vstack((zvect,z))
  return zvect, iter

def draw_sol(xx, n, beta, r, xsol):
  t = np.linspace(0,2*np.pi,200)
  xi = beta+r*np.exp(1.0j*t)
  plt.plot(xi.real, xi.imag,'c--')
  for i in range(n):
    plt.plot(xx[:,i].real, xx[:,i].imag,'bo')
  plt.plot(xsol.real, xsol.imag,'rx', ms=10)
  plt.grid('on')
  plt.gca().set_aspect('equal', adjustable='box') # set aspect ratio as 1:1
  plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/dk.pdf',dpi=300, bbox_inches='tight')
  plt.show()
  return 0

In [ ]:
import sympy
sympy.var('z')
a1=-2+1j
a2=-2-1j
a3=3+2j
a4=1
a5=2
f=sympy.expand((z-a1)*(z-a2)*(z-a3)*(z-a4)*(z-a5))
print(f)

In [ ]:
def f(x):
    return x**5.0-x**4.0*(2.0+2.0j)-x**3.0*(8.0+2.0j)+x**2*(8.0+10.0j)+x*(31.0+14.0j) - (30.0+20.0j)

n = 5
beta = 0.0
r = 5.0
#r=np.sqrt(13)
zvect, iter = durand_kerner(f, n, beta, r, 20, 1.0e-12)
zsol=np.array([-2.0+1.0j, -2-1.0j, 1.0, 2.0, 3.0+2.0j])
draw_sol(zvect, n, beta, r, zsol)
print(iter)
for i in range(n):
  print(zvect[iter,i])

# 3. Lagrange 補間多項式

図3.2（例3.2）

In [ ]:
from scipy.interpolate import lagrange

#補間関数
f_org = lambda x: x+np.sin(3.0*x)
#Lagrange補間を求める
a=0.0
b=4.0
n=10
x =np.linspace(a,b,n+1)
y = f_org(x)
f_lag = lagrange(x, y) #scipyのLagrange補間の関数を使う
#結果の表示
print(type(f_lag))
xnew =np.linspace(a,b,201)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, f_org(x), 'o', color='#ff7f00')
ax.plot(xnew, f_org(xnew), '-', color='#ff7f00')
ax.plot(xnew, f_lag(xnew), 'b-')
ax.legend(['f(x_i)','f(x)','pn(x)'], loc=(0.1, 0.8))
ax.grid(True)
ax.set_aspect('equal', adjustable='box')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/lag1n10.pdf',dpi=300, bbox_inches='tight')
plt.show()

図3.3（例3.3）

In [ ]:
from scipy.interpolate import lagrange

#補間関数
#f_org = lambda x: x+np.sin(3.0*x)
f_org = lambda x: 1.0 / (1.0 + x**2.0)
#Lagrange補間を求める
a=-5.0
b=5.0
n=12
x =np.linspace(a,b,n+1)
y = f_org(x)
f_lag = lagrange(x, y) #scipyのLagrange補間の関数を使う
#結果の表示
print(type(f_lag))
xnew =np.linspace(a,b,201)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, f_org(x), 'o',color='#ff7f00')
ax.plot(xnew, f_org(xnew), '-',color='#ff7f00')
ax.plot(xnew, f_lag(xnew), 'b-')
ax.legend(['f(x_i)','f(x)','pn(x)'], loc=(0.1, 0.8))
ax.grid(True)
ax.set_aspect('equal', adjustable='box')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/lag2n12.pdf',dpi=300, bbox_inches='tight')
plt.show()

図3.4（例3.4）

In [ ]:
from scipy.interpolate import lagrange

#補間関数
f_org = lambda x: 1.0 / (1.0 + x**2.0)
#Lagrange補間を求める
a=-5.0
b=5.0
n=20
#x =np.linspace(a,b,n+1)
x=np.empty(n+1)
for j in range(0,n+1,1):
    x[j] = 0.5*(b-a)*np.cos((j+0.5)*np.pi/(n+1))+0.5*(a+b);
y = f_org(x)
f_lag = lagrange(x, y) #scipyのLagrange補間の関数を使う
#結果の表示
print(type(f_lag))
xnew =np.linspace(a,b,201)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, f_org(x), 'o',color='#ff7f00')
ax.plot(xnew, f_org(xnew), '-',color='#ff7f00')
ax.plot(xnew, f_lag(xnew), 'b-')
ax.legend(['f(x_i)','f(x)','pn(x)'], loc=(0.1, 0.8))
ax.grid(True)
ax.set_aspect('equal', adjustable='box')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/lag3n20.pdf',dpi=300, bbox_inches='tight')
plt.show()

# 5. 直交多項式

図5.1（例5.6）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a = -1
b = 1
n = 400
x = np.linspace(a,b,n+1)

# チェビシェフ多項式 T_n(x) を漸化式で計算
def chebyshev_T(n, x):
    if n == 0:
        return np.ones_like(x)
    elif n == 1:
        return x
    else:
        T0 = np.ones_like(x)
        T1 = x
        for k in range(2, n + 1):
            T2 = 2 * x * T1 - T0
            T0, T1 = T1, T2
        return T1

# 1次から4次まで描画
plt.plot(x, chebyshev_T(1, x), '-',color='#CCFF33')
plt.plot(x, chebyshev_T(2, x), '-',color='#33FF66')
plt.plot(x, chebyshev_T(3, x), '-',color='#0099FF')
plt.plot(x, chebyshev_T(4, x), '-',color='#9966FF')
plt.legend(['T_1','T_2','T_3','T_4'])
plt.grid('on')
plt.gca().set_aspect('equal', adjustable='box')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/chebyshev.pdf',dpi=300, bbox_inches='tight')
plt.show()

図5.2（例5.7）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a = -1
b = 1
n = 400
x = np.linspace(a,b,n+1)

# ルジャンドル多項式 P_n(x)（漸化式）
def legendre_P(n, x):
    if n == 0:
        return np.ones_like(x)
    elif n == 1:
        return x
    else:
        P0 = np.ones_like(x)
        P1 = x
        for k in range(2, n + 1):
            P2 = ((2*k - 1) * x * P1 - (k - 1) * P0) / k
            P0, P1 = P1, P2
        return P1

plt.plot(x, legendre_P(1, x), '-',color='#CCFF33')
plt.plot(x, legendre_P(2, x), '-',color='#33FF66')
plt.plot(x, legendre_P(3, x), '-',color='#0099FF')
plt.plot(x, legendre_P(4, x), '-',color='#9966FF')
plt.legend(['P_1','P_2','P_3','P_4'])
plt.grid('on')
plt.gca().set_aspect('equal', adjustable='box')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/legendre.pdf',dpi=300, bbox_inches='tight')
plt.show()
plt.show()

# 6. Gauss型積分公式

表6.1（例6.4）

In [ ]:
#[0,1]上の直交多項式
def orth_poly2(x):
  return (6*x**2 - 6*x + 1)/6
def orth_poly3(x):
  return (20*x**3 - 30*x**2 + 12*x - 1) / 20

#Durand_Kernerで直交多項式の零点を求める
def durand_kerner2(f, maxiter, eps, xig):
  n=xig.shape[0]
  y=np.array([np.ones(n)]).T
  x = xig
  f_val = f(x)
  err = np.linalg.norm(f_val, ord=np.inf)
  iter = 0
  while err > eps and iter < maxiter:
    z = y@np.array([x])
    df_val = np.prod(z-z.T+np.eye(n,n), axis=0)
    if np.linalg.norm(df_val, ord=np.inf)<1.0e-15:
      print('denominator is almost zero')
      sys.exit(1)
    x -= f_val / df_val
    f_val = f(x)
    err = np.linalg.norm(f_val, ord=np.inf)
    iter += 1
  return x

#Gauss型積分を実行する
def gauss_quad(f, orth, n):
  #積分点の取得
  maxiter=100
  eps=1.0e-12
  nodes=durand_kerner2(orth, maxiter, eps, np.random.rand(n))
  #係数の取得
  V=np.vander(nodes,increasing=True).T
  b=np.array([1.0/(k+1.0) for k in range(nodes.shape[0])])
  weights=np.linalg.solve(V, b)
  return nodes, weights, np.dot(weights, f(nodes))

In [ ]:
k = 6
#被積分関数
a=0
b=1
def integrand(x):
  return x**k + x**(k-1) + 1
exact=1/(k+1)+1/k+1
#積分の計算
nodes, weights, val= gauss_quad(integrand, orth_poly3,3) #本当は事前に一度だけ実行しておけば良い
error=np.abs(exact-val)
print(f"value={val:.12f}")
print(f"error={np.abs(exact-val):.5e}")
print("nodes=",nodes)
print("weights=",weights)

# 7. 微分方程式の初期値問題

表7.1（例7.6）および表7.3，図7.5（例7.13）

In [ ]:
def euler(odefunc, t0, T, initial, N):
  t=np.linspace(t0, T, N+1)
  u=np.zeros(t.shape)
  h=(T-t0)/N
  u[0]=initial
  for n in range (N):
    u[n+1] = u[n] + h*odefunc(t[n],u[n])
  return h, t, u

In [ ]:
def heun(odefunc, t0, T, initial, N):
  t=np.linspace(t0, T, N+1)
  u=np.zeros(t.shape)
  h=(T-t0)/N
  u[0]=initial
  for n in range (N):
    tval = t[n]
    uval = u[n]
    k1 = odefunc(tval, uval);
    k2 = odefunc(tval+h, uval+h*k1)
    u[n+1] = uval + 0.5*h*(k1+k2)
  return h, t, u

In [ ]:
def rk(odefunc, t0, T, initial, N):
  t=np.linspace(t0, T, N+1)
  u=np.zeros(t.shape)
  h=(T-t0)/N
  u[0]=initial
  for n in range (N):
    tval = t[n]
    uval = u[n]
    k1 = odefunc(tval, uval);
    k2 = odefunc(tval+h/2.0, uval+h/2.0*k1)
    k3 = odefunc(tval+h/2.0, uval+h/2.0*k2)
    k4 = odefunc(tval+h, uval+h*k3)
    u[n+1] = uval + (h/6.0)*(k1 + 2.0*k2 + 2.0*k3 + k4)
  return h, t, u

In [ ]:
def OderError(Solver, odefunc, odeexact, t0, T, initial):
  NN=4
  kmax=8
  hv=np.zeros(kmax)
  ev=np.zeros(kmax)

  for k in range(kmax):
    h, t, u = Solver(odefunc, t0, T, initial, NN)
    exact=np.vectorize(odeexact)(t)
    error=u-exact
    ev[k] = np.linalg.norm(error,ord=np.inf)
    hv[k] = h
    NN = 2*NN

  rate=(np.log(ev[1:]) - np.log(ev[:-1])) / (np.log(hv[1:]) - np.log(hv[:-1]))

  return hv, ev, rate

In [ ]:
#右辺の関数
def func1(t, y):
 return 1.0 / (1.0 + y) - (t + 2.0 + (t + 1.0)**3) / ((t + 2.0)*(t + 1.0)**2)

#厳密解
def exact1(t):
  return 1.0 / (1.0 + t)

t0 = 0.0
T = 1.0
u0 = 1.0

hv_euler, ev_euler, rate_euler= OderError(euler, func1, exact1, t0, T, u0)
hv_heun, ev_heun, rate_heun= OderError(heun, func1, exact1, t0, T, u0)
hv_rk, ev_rk, rate_rk= OderError(rk, func1, exact1, t0, T, u0)

for i in range(rate_euler.shape[0]-1):
  print(f'{hv_euler[i+1]:.3f}, {ev_euler[i]:.5e}, {rate_euler[i]:.3f}, {ev_heun[i]:.5e}, {rate_heun[i]:.3f}, {ev_rk[i]:.5e}, {rate_rk[i]:.3f}')

#結果の描画（両対数目盛）
plt.figure(figsize=(3, 7))
plt.plot(hv_euler, ev_euler, 'bo-')
plt.plot(hv_heun, ev_heun, 'ro-')
plt.plot(hv_rk, ev_rk, 'go-')
plt.xscale('log')
plt.yscale('log')
plt.legend(['Euler','Heun','Runge-Kutta'])
plt.xlabel('h')
plt.ylabel('error')
plt.grid('on')
#plt.gca().set_aspect('equal', adjustable='box')
plt.show()

図7.6（例7.15）

In [ ]:
def rk4vec(odefunc, t0, T, initial, h):
  # if h>0, T must be > t0; if h<0, T must be < t0
  n=0
  unow=initial
  tnow=t0
  u=unow
  t=tnow
  while (T-t0)*(T-tnow-h) >= 0.0:
    k1 = odefunc(tnow, unow)
    k2 = odefunc(tnow+h/2, unow+(h/2)*k1)
    k3 = odefunc(tnow+h/2, unow+(h/2)*k2)
    k4 = odefunc(tnow+h, unow+h*k3)
    unow = unow + (h/6)*(k1 + 2*k2 + 2*k3 + k4)
    n += 1
    tnow = t0+n*h
    u = np.vstack((u, unow))
    t = np.append(t, tnow)
  return t, u

In [ ]:
def planet(t,u):
    R=(u[0]**2+u[1]**2)**(3/2)
    return np.array([u[2],u[3],-u[0]/R,-u[1]/R])

t0 = 0.0
T = 16.0*np.pi

k = 0.8
u0 = np.array([1-k, 0, 0, np.sqrt((1+k)/(1-k))])

h = 0.01
#t, u = eulervec(planet, t0, T, u0, h)
t, u = heunvec(planet, t0, T, u0, h)
#t, u = rk4vec(planet, t0, T, u0, h)

plt.plot(u[:,0],u[:,1],'b')
plt.xlabel('x')
plt.ylabel('y')
plt.grid('on')
plt.gca().set_aspect('equal', adjustable='box')
#plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/planet_euler.pdf',dpi=300, bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/planet_heun.pdf',dpi=300, bbox_inches='tight')
#plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/planet_rk.pdf',dpi=300, bbox_inches='tight')
plt.show()

図7.7（例7.16）

In [ ]:
def LotVol(t, u):
  a=1.0
  b=1.0
  c=1.0
  d=1.0
  return np.array([a*u[0]-b*u[0]*u[1],c*u[0]*u[1]-d*u[1]])

t0=0.0
T=40.0
u0=np.array([0.5,0.8])

h=0.01
t, u = rk4vec(LotVol, t0, T, u0, h)

plt.plot(t,u[:,0], color='#00bfff')
plt.plot(t,u[:,1], color='#ff8c00')
plt.xlabel('t')
plt.legend(['u','v'])
plt.grid('on')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/lv1.pdf',dpi=300, bbox_inches='tight')
plt.show()

図7.8（例7.16）

In [ ]:
def LotVol(t, u):
  a=1.0
  b=1.0
  c=1.0
  d=1.0
  return np.array([a*u[0]-b*u[0]*u[1],c*u[0]*u[1]-d*u[1]])

t0 = 0.0
T = 10.0
u0 = np.array([0.5,0.8])

h = 0.1
t, u = rk4vec(LotVol, t0, T, u0, h)

plt.plot(u[:,0],u[:,1],'b')
plt.xlabel('u')
plt.ylabel('v')
plt.grid('on')
plt.gca().set_aspect('equal', adjustable='box')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/lv2.pdf',dpi=300, bbox_inches='tight')
plt.show()

図7.9（例7.16）

In [ ]:
def LotVol(t, u):
  a=1.0
  b=1.0
  c=1.0
  d=1.0
  return np.array([a*u[0]-b*u[0]*u[1],c*u[0]*u[1]-d*u[1]])

t0=0.0
T=10.0

h=0.01
number=10
c=np.array([0,0])
length=1
for i in range(number):
  u0=c+length*np.random.rand(2)
  t, u = rk4vec(LotVol, t0, T, u0, h)
  plt.plot(u[:,0],u[:,1])

plt.xlabel('u')
plt.ylabel('v')
plt.grid('on')
plt.gca().set_aspect('equal', adjustable='box')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/lv.pdf',dpi=300, bbox_inches='tight')
plt.show()

図7.10（例7.17）

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

def Lorenz(t, u):
  a=10
  b=8/3
  c=28
  return np.array([a*(u[1]-u[0]), u[0]*(c-u[2])-u[1], u[0]*u[1]-b*u[2]])

t0=0.0
T=20.0

fig = plt.figure(figsize = (6,6))
ax = fig.add_subplot(111, projection='3d')

number=10
mag=3.0
h=0.001
for i in range(number):
  u0=-mag+2*mag*np.random.rand(3)
  t, u = rk4vec(Lorenz, t0, T, u0, h)
  ax.plot(u[:,0],u[:,1],u[:,2])

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')

ax.view_init(azim=-60)
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/lorenz.pdf',dpi=300, bbox_inches='tight')
plt.show()

図7.11（例7.18）

In [ ]:
def linsys(t,u):
  #A=np.array([[3,-1], [-1,2]])
  A=np.array([[2,3], [3,-1]])
  #A=np.array([[1,2], [4,5]])
  return np.dot(A,u)

t0=0.0
T=3.0

length=1
number=50
h=0.01
for i in range(number):
  u0=-length + 2*length*np.random.rand(2)
  # t>0 オレンジ
  t, u = rk4vec(linsys, t0, T, u0, h)
  plt.plot(u[:,0],u[:,1],color='#ff8c00')
  # t<0 シアン
  t, u = rk4vec(linsys, t0, -T, u0, -h)
  plt.plot(u[:,0],u[:,1],color='#00bfff')

plt.xlabel('x')
plt.ylabel('y')
plt.grid('on')
plt.xlim(-3*length,3*length)
plt.ylim(-3*length,3*length)
plt.gca().set_aspect('equal', adjustable='box')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/dyn.pdf',dpi=300, bbox_inches='tight')
plt.show()

図7.12（例7.19）

In [ ]:
def KerMcK(t, u):
  sigma=0.1
  rho=1.0
  return np.array([-sigma*u[0]*u[1],sigma*u[0]*u[1]-rho*u[1],rho*u[1]])

t0=0.0
T=20.0
u0=np.array([9.0,3.0,0.0]) # 初期値 [S(0), I(0), R(0)]

h=0.01
t, u = rk4vec(KerMcK, t0, T, u0, h)

plt.subplot(2, 1, 1)
plt.plot(t,u[:,0],'b')
plt.plot(t,u[:,1],'r')
plt.plot(t,u[:,2],'c')
plt.xlabel('t')
plt.legend(['S','I','R'])
plt.grid('on')

plt.subplot(2, 1, 2)
plt.plot(u[:,0],u[:,1],'b')
plt.xlabel('S')
plt.ylabel('I')
plt.grid('on')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/sir.pdf',dpi=300, bbox_inches='tight')
plt.show()

# 8. 固有値問題

図8.1

In [ ]:
def gershgorin(A):
    A = np.array(A)
    n = len(A)
    centers = np.diag(A)
    radii = np.sum(np.abs(A - np.diag(centers)), axis=1)
    bounds = np.array([
        centers.real - radii,
        centers.real + radii,
        centers.imag - radii,
        centers.imag + radii])

    fig, ax = plt.subplots()
    for center, radius in zip(centers, radii):
        circle = plt.Circle((center.real, center.imag), radius, color='b', fill=False)
        ax.add_artist(circle)

    print(f"実部の範囲: {bounds[0].min():.2f}, {bounds[1].max():.2f}")
    print(f"虚部の範囲: {bounds[2].min():.2f}, {bounds[3].max():.2f}")

    plt.xlim(bounds[0].min(), bounds[1].max())
    plt.ylim(bounds[2].min(), bounds[3].max())
    ax.set_aspect('equal')
    ax.set_xlabel('Real')
    ax.set_ylabel('Imaginary')
    ax.grid(True)
    plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/gershgorin.pdf',dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
#A = np.array([[30, 2, 3, 13], [5, 10, 11, 8],[9, 7, 6, 12],[4, 14, 15, 1]])
A = np.array([[1,2,3], [4,5,6],[7,8,9]])
gershgorin(A)

図8.2

In [ ]:
# diagonal entries of A
centers = np.array([1, 1j, -1, -1j])
# Gershgorin radii
radii = np.ones(4)

x = np.linspace(-2, 2, 1000)
y = np.linspace(-2, 2, 1000)
X, Y = np.meshgrid(x, y)
Z = X + 1j * Y
plt.figure(figsize=(6, 6))

# Cassini ovals
for i in range(4):
    for j in range(i + 1, 4):
        F = np.abs(Z - centers[i]) * np.abs(Z - centers[j])
        plt.contour(X, Y, F, levels=[radii[i] * radii[j]],
                    colors="red", linewidths=2)

# Gershgorin circles
theta = np.linspace(0, 2*np.pi, 1000)

for c, r in zip(centers, radii):
    plt.plot(c.real + r*np.cos(theta),
             c.imag + r*np.sin(theta),
             color="blue", linewidth=2)

plt.gca().set_aspect("equal")
plt.xlim(-2, 2)
plt.ylim(-2, 2)

plt.xlabel("Real")
plt.ylabel("Imaginary")
plt.grid(True, alpha=0.3)

plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/cassini.pdf',dpi=300, bbox_inches='tight')
plt.show()

例9.11

In [ ]:
from numpy import linalg as la
def power1(A, initial, tolerance):
  kmax = 100
  eval = np.array([])
  increment = np.array([])
  evec = np.copy(initial)
  k = 0
  x = initial / la.norm(initial, ord=2)
  muold = 0
  inc = tolerance + 1
  while k <= kmax and inc >= tolerance:
    y = A@x
    mu = np.vdot(y, x)
    inc = np.abs((mu - muold) / mu)
    eval = np.append(eval, mu)
    increment = np.append(increment, inc)
    evec = np.vstack((evec, x))
    x = y / la.norm(y, ord=2)
    muold = mu
    k += 1
  return k, eval, increment, evec[1:,:]


In [ ]:
n = 6 #全ウェブ数
H = np.array([
    [0,0,1/3,0,0,0],
    [1/2,0,1/3,0,0,0],
    [1/2,0,0,0,0,0],
    [0,0,0,0,1/2,1],
    [0,0,1/3,1/2,0,0],
    [0,0,0,1/2,1/2,0]
    ])
avec = np.zeros(n)
avec[1] = 1
S = H + (1/n)*np.outer(np.ones(n), avec)
print(f"H={H}")
print(f"a={avec}")
print(f"S={S}")

In [ ]:
alpha = 0.6
G = alpha*S + (1/n)*(1-alpha)*np.ones_like(S) # Google行列

tol = 1e-6  # 許容誤差
init = np.random.rand(6)# 初期ベクトル
iteration, eval, increment, evec = power1(G, init, tol)

print("反復, 固有値, 増分, 固有ベクトル")
for i in range(iteration):
  print(f"{i:d}, {eval[i]:.4f}, {increment[i]:.3e}, ",evec[i,:])

# PageRankベクトル
final = np.copy(evec[-1,:])
final = (1/np.sum(final))*final
print("PageRankベクトル",final)

# 10. 無制約最適化問題

図10.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L = 2.5
N = 300
x = np.linspace(-L, L, N)
y = np.linspace(-L, L, N)
X, Y = np.meshgrid(x, y)

Z = 2/5 - (1/10) * (5*X**2 + 5*Y**2 + 3*X*Y - X - 2*Y) * np.exp(-(X**2 + Y**2))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor="none")
ax.contour(X, Y, Z, zdir="z", offset=Z.min(), levels=20)
ax.set_zlim(Z.min(), Z.max())
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_xlim(-L, L)
ax.set_ylim(-L, L)
ax.view_init(elev=30, azim=-60)
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/example102.pdf',dpi=300, bbox_inches='tight')
plt.show()

図10.3

In [ ]:
from scipy.optimize import root

# p(x,y), q(x,y)
def p(x, y):
    return (-10*x**3-10*x*y**2-6*x**2*y+2*x**2+4*x*y+10*x+3*y-1)

def q(x, y):
    return (-10*y**3-10*y*x**2-6*y**2*x+4*y**2+2*x*y+10*y+3*x-2)

def F(z):
    x, y = z
    return np.array([p(x, y), q(x, y)])

x = np.linspace(-2, 2, 1000)
y = np.linspace(-2, 2, 1000)
X, Y = np.meshgrid(x, y)

P = p(X, Y)
Q = q(X, Y)

solutions = []
initial_points = np.linspace(-2, 2, 9)
for x0 in initial_points:
    for y0 in initial_points:
        sol = root(F, [x0, y0])
        if sol.success:
            xs, ys = sol.x
            if -2 <= xs <= 2 and -2 <= ys <= 2:
                # avoid duplicates
                if not any(np.linalg.norm(sol.x - s) < 1e-6 for s in solutions):
                    solutions.append(sol.x)
solutions = np.array(solutions)

plt.figure(figsize=(7, 7))
plt.contour(X, Y, P, levels=[0], colors="orange", linewidths=2)
plt.contour(X, Y, Q, levels=[0], colors="cyan", linewidths=2)
if len(solutions) > 0:
    plt.plot(
        solutions[:, 0],
        solutions[:, 1],
        "bo",
        markersize=5,
        label="stationary points"
    )
plt.xlabel(r"$x$")
plt.ylabel(r"$y$")
plt.gca().set_aspect("equal")
plt.xlim(-2, 2)
plt.ylim(-2, 2)
plt.grid(True, alpha=0.3)
plt.plot([], [], color="orange", lw=2, label=r"$p(x,y)=0$")
plt.plot([], [], color="cyan", lw=2, label=r"$q(x,y)=0$")
plt.legend()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/fig/stationary.pdf',dpi=300, bbox_inches='tight')
plt.show()
print("stationary points:")
for s in solutions:
    print(f"({s[0]: .10f}, {s[1]: .10f})")

例10.2

In [ ]:
# Newton method
import sys

def newton_mult(f, dfdx, x, eps, maxiter):
  iter = 0
  fval = f(x)
  xvect = np.copy(x)
  fvect = np.copy(fval)
  magnitude = np.linalg.norm(fval, ord=np.inf)
  while magnitude > eps and iter < maxiter:
    dfdxval = dfdx(x)
    if np.abs(np.linalg.det(dfdxval)) < 1.0e-16:
      print('iteration =', iter)
      print('Jacobi matrix is almost singular for x = ', x)
      sys.exit(1)
    w = np.linalg.solve(dfdxval, fval)
    x -= w
    fval = f(x)
    magnitude = np.linalg.norm(fval, ord=np.inf)
    iter += 1
    xvect=np.vstack((xvect,x))
    fvect=np.vstack((fvect,fval))
  return xvect, fvect, iter

In [ ]:
def f_pq(z):
  x, y = z
  p = (-10*x**3 - 10*x*y**2 - 6*x**2*y + 2*x**2 + 4*x*y + 10*x + 3*y - 1)
  q = (-10*y**3 - 10*y*x**2 - 6*y**2*x+ 4*y**2 + 2*x*y + 10*y + 3*x - 2)
  return np.array([p, q])

def Jac_f_pq(z):
  x, y = z
  p = (-10*x**3 - 10*x*y**2 - 6*x**2*y + 2*x**2 + 4*x*y + 10*x + 3*y - 1)
  q = (-10*y**3 - 10*y*x**2 - 6*y**2*x+ 4*y**2 + 2*x*y + 10*y + 3*x - 2)
  px = -30*x**2 - 10*y**2 - 12*x*y + 4*x + 4*y + 10
  py = -20*x*y - 6*x**2 + 4*x + 3
  qx = -20*x*y - 6*y**2 + 2*y + 3
  qy = -30*y**2 - 10*x**2 - 12*x*y + 8*y + 2*x + 10
  return np.array([[px, py],[qx, qy]])

In [ ]:
# p と q の交点を Newton method で計算

# initial guess
z = np.array([-0.5,-0.5])

# iteration
kmax = 50
eps = 1e-12
xvect, fvect, iter = newton_mult(f_pq, Jac_f_pq, z, eps, kmax)
# results
print(f'max number of iterations: {kmax:d}')
print('iteration',iter)
for i in range(iter):
  print(i,xvect[i],np.linalg.norm(fvect[i], ord=np.inf))

In [ ]:
def f102(z):
  x, y = z
  return 2/5 - (1/10)*(5*x**2 + 5*y**2 + 3*x*y - x - 2*y)*np.exp(-(x**2 + y**2))

def grad_f102(z):
  x, y = z
  e = np.exp(-(x**2 + y**2))
  p = (-10*x**3 - 10*x*y**2 - 6*x**2*y + 2*x**2 + 4*x*y + 10*x + 3*y - 1)
  q = (-10*y**3 - 10*y*x**2 - 6*y**2*x+ 4*y**2 + 2*x*y + 10*y + 3*x - 2)
  return -0.1 * e * np.array([p, q])

def hessian_f102(z):
  x, y = z
  e = np.exp(-(x**2 + y**2))
  p = (-10*x**3 - 10*x*y**2 - 6*x**2*y + 2*x**2 + 4*x*y + 10*x + 3*y - 1)
  q = (-10*y**3 - 10*y*x**2 - 6*y**2*x+ 4*y**2 + 2*x*y + 10*y + 3*x - 2)
  px = -30*x**2 - 10*y**2 - 12*x*y + 4*x + 4*y + 10
  py = -20*x*y - 6*x**2 + 4*x + 3
  qx = -20*x*y - 6*y**2 + 2*y + 3
  qy = -30*y**2 - 10*x**2 - 12*x*y + 8*y + 2*x + 10
  H11 = -0.1 * e * (px - 2*x*p)
  H12 = -0.1 * e * (py - 2*y*p)
  H21 = -0.1 * e * (qx - 2*x*q)
  H22 = -0.1 * e * (qy - 2*y*q)
  return np.array([[H11, H12],[H21, H22]])

In [ ]:
za = np.array([-0.59544293,-0.71610852])
zb = np.array([-0.52457982,0.97455382])
zc = np.array([0.04438578,0.17921078])
zd = np.array([0.94288967,-0.3701998])
ze = np.array([0.88732605,0.63950342])

print(np.linalg.eig(hessian_f102(za))[0])
print(np.linalg.eig(hessian_f102(zb))[0])
print(np.linalg.eig(hessian_f102(zc))[0])
print(np.linalg.eig(hessian_f102(zd))[0])
print(np.linalg.eig(hessian_f102(ze))[0])

In [ ]:
print(f102(za))
print(f102(ze))

In [ ]:
# Newton methodでの計算

# initial guess
z = np.array([-0.5,-0.5])

# iteration
kmax = 50
eps = 1e-16
xvect, fvect, iter = newton_mult(grad_f102, hessian_f102, z, eps, kmax)
# results
print(f'max number of iterations: {kmax:d}')
print('iteration',iter)
for i in range(iter):
  print(i,xvect[i],np.linalg.norm(fvect[i], ord=np.inf))

In [ ]:
# 最急降下法（Armijo条件）
def steepest_descent(f, grad_f, z0, sigma, rho, tol, max_iter):
  alpha0 = 1.0
  z = np.array(z0, dtype=float)
  fval = f(z)
  xvect = np.copy(z)
  fvect = np.copy(fval)
  for k in range(max_iter):
    g = grad_f(z)
    if np.linalg.norm(g) < tol:
      return xvect, fvect, k
    d = -g
    alpha = alpha0
    while f(z + alpha*d) > f(z) + sigma*alpha*np.dot(g, d):
      alpha *= rho
    z = z + alpha*d
    fval = f(z)
    xvect = np.vstack((xvect,z))
    fvect = np.vstack((fvect,fval))
  return xvect, fvect, max_iter

In [ ]:
#最急降下法での計算

#initial guess
z0 = np.array([1.0,1.0])

#iteration
rho = 0.25
sigma = 0.05
tol = 1e-8
max_iter = 10000
xvect, fvect, it_min = steepest_descent(f102, grad_f102, z0, sigma, rho, tol, max_iter)

#results
print(f'max number of iterations: {kmax:d}')
print('iteration',it_min)
for i in range(it_min):
  print(i,xvect[i],fvect[i])

# B. 計算機における数の表現

In [1]

In [ ]:
10**50 + 812 - 10**50 + 10**35 - 12 - 10**35

In [2]

In [ ]:
10.0**50 + 812.0 - 10.0**50 + 10.0**35 - 12.0 - 10.0**35

In [3]

In [ ]:
eps=2**-52
print(eps)
print(1.0+(1.0/4.0)*eps)
print(1.0+(1.0/2.0)*eps)
print(1.0+(3.0/4.0)*eps)
print(1.0+eps)

In [7]

In [ ]:
x=192119201.0
y=35675640.0
a=(1682*x*y**4 + 3*x**3 + 29*x*y**2 - 2*x**5 + 832)/107751
print(a)

In [ ]:
x=192119201
y=35675640
a=(1682*x*y**4 + 3*x**3 + 29*x*y**2 - 2*x**5 + 832)/107751
b=1682*x*y**4 + 3*x**3 + 29*x*y**2
c=-2*x**5 + 832
print(a)
print(b)
print(c)